# Chapter 01: Dataset Loading & Validation

## Engineering Question
> Why is understanding the dataset loading and validation phase critical before building an intrusion detection system?

---

### Objective
The objective of this notebook is to load the NSL-KDD training and testing datasets using our centralized, standardized dataset loading utility, validate their structures, and explore the high-level features and classes without executing any preprocessing, encoding, or model fitting. This establishes a clean, validated data checkpoint that guarantees downstream processing starts from a trusted state.

## Background / Theory

### Dataset Overview
The NSL-KDD dataset is a curated benchmark dataset for evaluating network intrusion detection systems. It represents a refinement of the historic KDD Cup 1999 dataset, containing millions of network connection records categorized as normal or attacks. Attacks are split into four main families:
1. **DoS (Denial of Service)**: Attacks attempting to exhaust server resources (e.g. `neptune`, `smurf`).
2. **Probe**: Surveillance and scanning attempts to gather network information (e.g. `satan`, `ipsweep`).
3. **U2R (User to Root)**: Privilege escalation attempts to gain superuser access (e.g. `buffer_overflow`, `rootkit`).
4. **R2L (Remote to Local)**: Unauthorized access attempts from remote systems (e.g. `guess_passwd`, `warezmaster`).

### Why NSL-KDD?
The original KDD99 dataset suffered from severe systemic flaws. Specifically, it contained millions of duplicate records which biased classifiers towards frequent attacks and frequent normal records. NSL-KDD addresses these issues by:
- **Eliminating Duplicate Records**: Preventing the model from memorizing duplicates, ensuring reliable evaluation.
- **Balancing Difficulty Levels**: Selecting samples such that classifiers must learn general boundaries rather than memorizing high-frequency features.
- **Providing Reasonable Sizes**: Making the dataset computationally feasible for rapid iteration while maintaining diverse attack types.

### Unsupervised Anomaly Detection Applicability
In real-world networks, cyber attacks evolve continuously. Supervised models trained on specific attack patterns fail to detect novel, unseen zero-day attacks. Unsupervised anomaly detection algorithms (such as Isolation Forest, DBSCAN, or Autoencoders) model the boundaries of normal baseline traffic and flag any deviation as anomalous, making them far more robust to zero-day scenarios.

### Dataset Assumptions
Unsupervised algorithms make fundamental assumptions about the data structure:
- **Isolation Forest**: Assumes anomalies are rare and different (highly isolated in space).
- **DBSCAN**: Assumes normal records form dense clusters, while anomalies form sparse, low-density noise points.
- **Autoencoder**: Assumes a neural network trained solely on normal traffic will fail to reconstruct attack patterns, producing high reconstruction errors.

## Dataset Files
- `KDDTrain+.txt`: The primary training dataset containing 125,973 records.
- `KDDTest+.txt`: The testing dataset containing 22,544 records. It contains 17 novel attack categories not present in the training set, serving as an excellent test for generalization.

## Workflow Diagram

```text
  [Raw Text Files] (KDDTrain+.txt / KDDTest+.txt)
         │
         ▼
  [load_dataset()] ──► Apply DATASET_COLUMNS Schema
         │
         ▼
  [Drop Column] ─────► Remove 'difficulty' (Prevents Data Leakage)
         │
         ▼
  [validate_dataset] ► Run Sanity Checks (Not Empty, Check Label Column)
         │
         ▼
  [Outputs] ────────► Clean training & testing DataFrames
```

## Imports

All imports originate from standard libraries or our modularized project backend (`src` / `configs`). This enforces a single source of truth for the codebase.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Ensure project root is in path for imports
sys.path.append(os.path.abspath(".." if ".." in sys.path else ".."))

from configs import config
from src.data.dataset import load_train_data, load_test_data, dataset_summary, validate_dataset

# Apply standard Plotly styling
pio.templates.default = config.PLOT_TEMPLATE

## Dataset Loading

We load the training and testing datasets using the modularized functions. This ensures that the schema columns and file path definitions are loaded from config constants.

In [2]:
train_df = load_train_data()
test_df = load_test_data()

print(f"Train dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")
print("Dataset files loaded successfully!")

Train dataset shape: (125973, 42)
Test dataset shape: (22544, 42)
Dataset files loaded successfully!


## Dataset Validation

Validate that the loaded datasets conform to standard rules: they must not be empty, must be loaded as pandas DataFrames, and must contain the required target column (`label`).

In [3]:
validate_dataset(train_df)
validate_dataset(test_df)
print("Training dataset validation passed successfully.")
print("Testing dataset validation passed successfully.")

Training dataset validation passed successfully.
Testing dataset validation passed successfully.


## Dataset Summary

We call `dataset_summary` to inspect the general properties of both splits.

In [4]:
train_sum = dataset_summary(train_df)
test_sum = dataset_summary(test_df)

print("=== Training Dataset Summary ===")
print(f"Rows Count: {train_sum['rows']}")
print(f"Columns Count: {train_sum['columns']}")
print(f"Missing Values: {train_sum['missing_values']}")
print(f"Duplicate Rows: {train_sum['duplicate_rows']}")
print(f"Unique Target Labels: {train_sum['attack_classes']}")

print("\n=== Testing Dataset Summary ===")
print(f"Rows Count: {test_sum['rows']}")
print(f"Columns Count: {test_sum['columns']}")
print(f"Missing Values: {test_sum['missing_values']}")
print(f"Duplicate Rows: {test_sum['duplicate_rows']}")
print(f"Unique Target Labels: {test_sum['attack_classes']}")

=== Training Dataset Summary ===
Rows Count: 125973
Columns Count: 42
Missing Values: 0
Duplicate Rows: 0
Unique Target Labels: 23

=== Testing Dataset Summary ===
Rows Count: 22544
Columns Count: 42
Missing Values: 0
Duplicate Rows: 0
Unique Target Labels: 38


## Feature Overview

Identify the data types and column splits (categorical vs numerical) within our feature space.

In [5]:
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
# Remove target column from categories list
if 'label' in categorical_cols:
    categorical_cols.remove('label')
numerical_cols = train_df.select_dtypes(exclude=['object']).columns.tolist()

print(f"Categorical columns in training set: {categorical_cols}")
print(f"Numerical columns count: {len(numerical_cols)}")

Categorical columns in training set: ['protocol_type', 'service', 'flag']
Numerical columns count: 38


## Target Classes

Let's look at the class distribution inside the training and test sets. We will visualize these distributions using interactive Plotly bar charts.

In [6]:
# Calculate train class distribution
train_counts = train_df['label'].value_counts().reset_index()
train_counts.columns = ['Label', 'Count']

fig_train = px.bar(
    train_counts,
    x='Label',
    y='Count',
    title='Training Target Class Distribution',
    labels={'Label': 'Traffic Category', 'Count': 'Records Count'},
    color='Label'
)
fig_train.update_layout(showlegend=False, width=750, height=450)
fig_train.show()

Let's also visualize the test set class distribution to inspect labels.

In [7]:
# Calculate test class distribution
test_counts = test_df['label'].value_counts().reset_index()
test_counts.columns = ['Label', 'Count']

fig_test = px.bar(
    test_counts,
    x='Label',
    y='Count',
    title='Testing Target Class Distribution',
    labels={'Label': 'Traffic Category', 'Count': 'Records Count'},
    color='Label'
)
fig_test.update_layout(showlegend=False, width=750, height=450)
fig_test.show()

## Initial Observations

- **General Statistics**: The training dataset contains exactly 125,973 records, while the testing dataset has 22,544 records.
- **Feature Schema**: Excluding the target column, the dataset contains 41 features (3 categorical and 38 numerical).
- **Target Label Count**: There are 23 unique labels in the training set, but 38 unique labels in the test set. This highlights a significant challenge: the test set contains 15 novel attack types that the model has never encountered during training (such as `mscan`, `saint`, `httptunnel`).
- **Class Balance**: Normal traffic accounts for 53.45% of the training set (67,343 records) and 43.08% of the test set (9,711 records). This contradicts standard anomaly detection assumptions where anomalies are extremely rare (e.g. <1%).

## Engineering Notes

### Why was the difficulty column removed?
The `difficulty` column is a scoring support column introduced by the creators of NSL-KDD. It tracks how many out of 21 baseline classifiers correctly predicted the record. While highly informative for analyzing classifier difficulty, it is not a raw network packet metric. Leaving it in the feature space would introduce severe data leakage because the column acts as a proxy for the label.

### Why do target labels remain strings at this stage?
To adhere to single-responsibility architecture, the loader is purely responsible for reading and validating dataset schemas. Transforming string labels (e.g. converting different attack types to a binary `0/1` space) is a preprocessing task that must be handled downstream to prevent mixing loader logic with ML transformations.

### How does this scale to production logs?
In a production system, raw logs from firewalls or NetFlow probes will not arrive formatted as clean CSVs. A streaming ingestion tool (such as Apache Kafka or AWS Kinesis) must capture traffic packets, compute sliding-window metrics (like service frequencies and count rates), discard metadata columns, validate datatypes, and then load features into a prediction queue.

## Interview Questions

1. **What are the main systemic flaws of the original KDD Cup 1999 dataset, and how does NSL-KDD address them?**
   * *Guideline*: Emphasize the removal of duplicate records in both training and test datasets. Explain how duplicate records skew classification accuracies, making models overfit to high-frequency duplicate attacks, and how NSL-KDD resolves this bias.

2. **Why does the presence of 15 novel attack types in the testing dataset justify the use of unsupervised anomaly detection over supervised classification?**
   * *Guideline*: Explain that supervised models learn closed-world classifications and are blind to unseen categories. Unsupervised models learn normal baseline boundary features and flag any deviations, enabling detection of novel attack families.

3. **What is data leakage, and why would keeping the 'difficulty' column represent a leakage risk?**
   * *Guideline*: Define data leakage as the inclusion of information in training data that would not be available during real-world inference. Since 'difficulty' is calculated based on prior model predictions, it is a target proxy and violates independent-feature assumptions.

4. **The NSL-KDD dataset has an attack rate of ~46.5% in training. How does this affect standard unsupervised anomaly detection assumptions?**
   * *Guideline*: Explain that models like Isolation Forest assume anomalies are sparse and rare. An attack rate of 46.5% violates this assumption, requiring us to carefully manage contamination parameters or restrict training strictly to normal baseline traffic (as in Autoencoders).

5. **Why is it a best practice to decouple dataset loading/validation logic from data preprocessing/encoding pipelines?**
   * *Guideline*: Decoupling guarantees modularity, maintainability, and clean test coverage. It prevents data loading functions from becoming monolithic, letting developers test loading structures independently of preprocessing encoders.

## Key Takeaways
- Decoupled loading logic ensures files load cleanly without side effects.
- The test split contains novel attacks, demonstrating the value of unsupervised boundary detectors.
- The dataset contains no missing values, but possesses significant label shift between training and test sets.

## Future Improvements
- **Schema Validation Classes**: Introduce Pydantic validation classes in `src.data.dataset` to strictly check feature column types at runtime during production deployments.

## Conclusion

We have loaded, validated, and summarized the NSL-KDD dataset splits. We now understand the structure of its features and the distribution of attack categories.

## Next Notebook

Proceed to the next chapter: [Exploratory Data Analysis](file:///c:/Projects/Network%20anomoly%20detection/notebooks/02_exploratory_data_analysis.ipynb)